# Sports14 Text/Image Feature Extraction

In [2]:

import os
import numpy as np
import pandas as pd

## Load text data

In [3]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
file_name = 'meta-toys.csv'

meta_file = os.path.join(file_path, file_name)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

data loaded!
shape: (11924, 10)


,itemID,asin,description,title,price,salesRank,imUrl,brand,categories,related
0,0,B001RNFQNK,Classic Electronic Catch Phrase with all new p...,Hasbro Electronic Catch Phrase,29.99,{'Toys & Games': 3095},http://ecx.images-amazon.com/images/I/413BcULs...,Parker Brothers,"[['Toys & Games', 'Games', 'Handheld Games']]","{'also_bought': ['B00H4OKN48', 'B001RNC0VG', '..."
1,1,158978068X,NaN,Gloom,17.28,NaN,http://ecx.images-amazon.com/images/I/51H3uVu1...,Atlas,"[['Toys & Games', 'Games', 'Board Games']]",NaN
2,2,B004S8F7QM,Cards Against Humanity is a party game for hor...,Cards Against Humanity,25.00,{'Toys & Games': 1},http://ecx.images-amazon.com/images/I/41164wOO...,NaN,"[['Toys & Games', 'Games', 'Card Games']]","{'also_bought': ['B005JFNE8G', 'B008JNPBYK', '..."


In [4]:

# sentences: title + brand + category + description | All have title + description

title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['description'].isnull()]
print(desc_na_df.shape)

na_df = df[df['description'].isnull() & df['title'].isnull()]
print(na_df.shape)

na3_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull()]
print(na3_df.shape)

na4_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull() & df['categories'].isnull()]
print(na4_df.shape)

(59, 10)
(798, 10)
(34, 10)
(34, 10)
(0, 10)


In [5]:

df[desc_str] = df[desc_str].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")


In [6]:
import json


def replace_empty_with_none(value):
    if value.strip() == "":
        return "None"
    return value

# Apply the function to relevant columns
df['title'] = df['title'].apply(replace_empty_with_none)
df['description'] = df['description'].apply(replace_empty_with_none)
df['brand'] = df['brand'].apply(replace_empty_with_none)
df['categories'] = df['categories'].apply(replace_empty_with_none)

# Convert DataFrame to a list of dictionaries, each representing a beauty product
beauty_products = df.apply(lambda row: {
    "itemID": row['itemID'],
    "title": row['title'],
    "description": row['description'],
    "brand": row['brand'],
    "categories": row['categories']
}, axis=1).tolist()

# Define the output JSON file path
output_file_path = os.path.join(file_path, 'toys_products.json')

# Write the list of dictionaries to a JSON file
with open(output_file_path, 'w', encoding='utf-8') as json_file:
    json.dump(beauty_products, json_file, indent=4, ensure_ascii=False)

print(f'JSON file saved at: {output_file_path}')


JSON file saved at: ./toys_products.json


In [7]:
# Read existing data if file exists, then write updated data in one line format
output_file_path = os.path.join(file_path, 'toys_products.json')

if os.path.exists(output_file_path):
    with open(output_file_path, 'r', encoding='utf-8') as json_file:
        existing_data = json_file.read()
else:
    existing_data = ""
output_file_path = "step1_item_positive_prompt.json"
# Write updated data (without itemID) in one-line format
with open(output_file_path, 'w', encoding='utf-8') as json_file:
    for product in beauty_products:
        # Remove the itemID key if it exists
        product_without_itemid = {key: value for key, value in product.items() if key != "itemID"}
        
        prompt_data = {
            "prompt": f"BASIC INFORMATION: \n{json.dumps(product_without_itemid, ensure_ascii=False)}",
        }
        
        # Write the product data in one line
        json_file.write(json.dumps(prompt_data, ensure_ascii=False) + "\n")

print(f'JSON file saved at: {output_file_path}')

JSON file saved at: step1_item_positive_prompt.json
